# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Separar o Rio Grande do Sul pelas Regiões Funcionais de Planejamento

**Objetivo:** unir a malha municipal do IBGE ao arquivo `municipios_regioes_funcionais_rs.csv` e produzir camadas geográficas que separem o território do estado pelas 9 Regiões Funcionais de Planejamento.

**Produto:** um GeoPackage com duas camadas (municípios classificados e polígonos das Regiões Funcionais), um GeoJSON para mapa web, um CSV de atributos, um mapa HTML pronto para abrir no navegador e um registro JSON com as fontes e as decisões de preparação.

**Percurso:** (A) baixar malha e tabela com registro de procedência; (B) inspecionar e limpar; (C) cruzar por nome normalizado, medindo a cobertura; (D) dissolver em regiões; (E) salvar, reabrir e conferir no mapa.

Este roteiro segue o fluxo de `aula03_07_Obtencao_Dados_Geograficos.ipynb`: download com procedência, inspeção antes da transformação, junção validada e reabertura dos arquivos salvos.

## 1. Leia antes de rodar: o que este dado permite

Um mapa por Região Funcional precisa de duas coisas: a **malha** com os polígonos dos municípios e uma **tabela** que diga a que região cada município pertence. A malha do IBGE cobre o estado inteiro. A tabela, não.

O arquivo `municipios_regioes_funcionais_rs.csv` mapeia **COREDE → Região Funcional** e, em cada linha, lista apenas os *principais municípios* daquele COREDE. São 27 linhas para 27 COREDEs, citando 85 municípios distintos — de um estado que tem 497.

| | |
|---|---|
| Municípios do RS na malha do IBGE | **497** |
| Nomeados no arquivo de Regiões Funcionais | 85 |
| Com região atribuída, após excluir o ambíguo | **84** (16,9%) |
| Sem correspondência | **413** (83,1%) |

Dos 85 nomeados, 84 entram na classificação: Frederico Westphalen aparece em duas Regiões Funcionais diferentes e é excluído (seção 5). Em área, os 84 correspondem a **39% do território do estado** — os polos concentram municípios extensos da metade sul e da fronteira oeste.

**Consequência para o mapa:** os 84 municípios classificados aparecem coloridos; os 413 restantes ficam em **cinza**, rotulados como sem correspondência no arquivo. O resultado mostra os **polos** de cada região, não a extensão territorial completa das regiões.

Este notebook **não inventa** a região dos municípios ausentes. Atribuí-los por proximidade produziria um mapa cheio, mas com fronteiras que não são as oficiais dos COREDEs. Para o mapa completo e correto seria preciso o mapeamento oficial município → COREDE, publicado pela SEPLAG-RS, que não está neste repositório.

A seção 6 mede essa cobertura região por região, para que a limitação fique explícita no próprio resultado.

## 2. Preparação e pastas

As saídas ficam em `dados/regioes_funcionais`, criadas a partir da pasta onde o notebook está.

```text
dados/regioes_funcionais/
├── brutos/          # cópias baixadas, com registro de procedência
└── preparados/      # resultados deste roteiro
```

Instale as dependências no mesmo kernel que executa o notebook.

In [ ]:
# Se necessário, descomente:
# %pip install geopandas pandas folium mapclassify

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlopen, Request
import hashlib
import json
import re
import unicodedata

import geopandas as gpd
import pandas as pd
import folium

PASTA_TRABALHO = Path("dados") / "regioes_funcionais"
PASTA_BRUTOS = PASTA_TRABALHO / "brutos"
PASTA_PREPARADOS = PASTA_TRABALHO / "preparados"
for pasta in (PASTA_BRUTOS, PASTA_PREPARADOS):
    pasta.mkdir(parents=True, exist_ok=True)

print("Pasta de trabalho:", PASTA_TRABALHO.resolve())
print("geopandas", gpd.__version__, "| pandas", pd.__version__, "| folium", folium.__version__)

### Download com reutilização e registro de procedência

`baixar` grava os bytes recebidos e registra URL, data UTC, tamanho e SHA-256. O hash identifica a cópia exata usada; não atesta, sozinho, a qualidade do dado. Um arquivo já existente com registro é conferido antes de ser reutilizado, e o download passa por um `.part` para que uma interrupção não deixe um arquivo truncado com o nome final.

In [ ]:
def sha256(arquivo):
    resumo = hashlib.sha256()
    with Path(arquivo).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()


def baixar(url, destino):
    """Baixa uma vez e reutiliza, registrando procedência em <arquivo>.fonte.json."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    registro = destino.with_suffix(destino.suffix + ".fonte.json")

    if destino.exists():
        if registro.exists():
            meta = json.loads(registro.read_text(encoding="utf-8"))
            if meta["url"] != url or meta["sha256"] != sha256(destino):
                raise ValueError(f"Cópia local não confere com o registro: {destino}")
        print("Reutilizando:", destino.name)
        return destino

    parcial = destino.with_suffix(destino.suffix + ".part")
    pedido = Request(url, headers={"User-Agent": "IA001-notebook"})
    with urlopen(pedido, timeout=180) as resposta, parcial.open("wb") as saida:
        while bloco := resposta.read(1024 * 1024):
            saida.write(bloco)
    parcial.replace(destino)

    registro.write_text(json.dumps({
        "url": url,
        "baixado_em_utc": datetime.now(timezone.utc).isoformat(),
        "bytes": destino.stat().st_size,
        "sha256": sha256(destino),
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Baixado:", destino.name, f"({destino.stat().st_size / 1_048_576:.1f} MB)")
    return destino

## 3. As duas fontes

| Fonte | O que traz | Origem |
|---|---|---|
| `RS_Municipios_2022.zip` | Polígonos dos municípios do RS, com código e nome do IBGE | [Malhas municipais, IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_municipais/municipio_2022/UFs/RS/) |
| `municipios_regioes_funcionais_rs.csv` | COREDE → Região Funcional, com os principais municípios de cada COREDE | Repositório do grupo, a partir da [SEPLAG-RS](https://planejamento.rs.gov.br/coredes) |

A malha é de 2022 e o arquivo de Regiões Funcionais não tem ano de referência declarado; a junção é feita por nome, e não por código, porque o CSV não traz o código do IBGE.

In [ ]:
URL_MALHA = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/"
    "malhas_municipais/municipio_2022/UFs/RS/RS_Municipios_2022.zip"
)
URL_REGIOES = (
    "https://raw.githubusercontent.com/valandro/ufrgs-spec-ai/main/"
    "AI001-analise-visualizacao-dados/educacao_rs/datasets/municipios_regioes_funcionais_rs.csv"
)

ARQUIVO_MALHA = baixar(URL_MALHA, PASTA_BRUTOS / "RS_Municipios_2022.zip")
ARQUIVO_REGIOES = baixar(URL_REGIOES, PASTA_BRUTOS / "municipios_regioes_funcionais_rs.csv")

## 4. Inspecionar a malha antes de transformar

Duas coisas precisam de atenção nesta malha:

1. Ela tem **499 feições, mas o RS tem 497 municípios**. As duas sobrando são corpos d'água — Lagoa dos Patos e Lagoa Mirim — que o IBGE distribui na mesma camada, com códigos reservados `4300001` e `4300002`. Somá-las como municípios inflaria área e contagem.
2. O CRS é `EPSG:4674` (SIRGAS 2000), em graus. **Não é uma projeção métrica**: calcular área diretamente nele daria um número sem significado. A área é calculada mais adiante, em UTM.

Os códigos são preservados como texto: `4300034` não é um número a ser somado.

In [ ]:
malha = gpd.read_file(f"zip://{ARQUIVO_MALHA}")
print("Feições:", len(malha), "| CRS:", malha.crs)
print("Colunas:", malha.columns.tolist())

# Códigos reservados do IBGE para corpos d'água nesta camada — não são municípios.
CODIGOS_AGUA = {"4300001", "4300002"}
malha["CD_MUN"] = malha["CD_MUN"].astype(str)
agua = malha[malha["CD_MUN"].isin(CODIGOS_AGUA)]
print("\nRemovidos por não serem municípios:")
print(agua[["CD_MUN", "NM_MUN", "AREA_KM2"]].to_string(index=False))

municipios = malha[~malha["CD_MUN"].isin(CODIGOS_AGUA)][
    ["CD_MUN", "NM_MUN", "geometry"]
].copy()

if len(municipios) != 497:
    raise ValueError(f"Esperados 497 municípios, encontrados {len(municipios)}.")
if municipios["CD_MUN"].duplicated().any():
    raise ValueError("Há códigos de município repetidos na malha.")
if municipios.geometry.is_empty.any() or not municipios.geometry.is_valid.all():
    raise ValueError("Há geometrias vazias ou inválidas na malha.")

print(f"\nMunicípios válidos: {len(municipios)}")
display(municipios.drop(columns="geometry").head())

## 5. Ler e desmembrar o arquivo de Regiões Funcionais

O CSV tem uma linha por COREDE, e os municípios vêm todos numa única célula, separados por vírgula. Para cruzar com a malha é preciso **desmembrar** essa coluna: uma linha por município.

Três detalhes do arquivo exigem tratamento, e nenhum deles é evidente ao abrir a planilha:

- **BOM no início.** O arquivo começa com marca de ordem de byte, que faria o nome da primeira coluna ser lido como `﻿Regiao_Funcional_ID`. Resolve-se com `encoding="utf-8-sig"`.
- **Marcação `(parcial)`.** Bagé aparece duas vezes, e uma delas está marcada assim. Tratamos a linha marcada como menção secundária, não como atribuição principal.
- **Ambiguidade real.** Frederico Westphalen aparece em dois COREDEs de **Regiões Funcionais diferentes**, sem nada que permita desempatar. É excluído, e o código informa quando isso acontece. Vacaria também aparece duas vezes, mas os dois COREDEs estão na mesma RF — não há conflito no nível que interessa aqui.

In [ ]:
regioes = pd.read_csv(ARQUIVO_REGIOES, encoding="utf-8-sig")
print(f"COREDEs: {len(regioes)} | Regiões Funcionais: {regioes['Regiao_Funcional_ID'].nunique()}")
display(regioes.head(3))

linhas = []
for _, linha in regioes.iterrows():
    for exemplo in str(linha["Principais_Municipios_Exemplos"]).split(","):
        nome = exemplo.strip()
        linhas.append({
            "nome_csv": re.sub(r"\(parcial\)", "", nome).strip(),
            "corede": linha["COREDE"],
            "rf": int(linha["Regiao_Funcional_ID"]),
            # "(parcial)" marca menção secundária, não atribuição principal
            "parcial": "(parcial)" in nome,
        })
polos = pd.DataFrame(linhas)
print(f"\nMenções a municípios: {len(polos)} | distintos: {polos['nome_csv'].nunique()}")

polos = polos[~polos["parcial"]]
rfs_por_municipio = polos.groupby("nome_csv")["rf"].nunique()
ambiguos = rfs_por_municipio[rfs_por_municipio > 1].index.tolist()
if ambiguos:
    print(f"Excluídos por Região Funcional ambígua no arquivo: {ambiguos}")
polos = polos[~polos["nome_csv"].isin(ambiguos)].drop_duplicates("nome_csv")
print(f"Municípios com Região Funcional definida: {len(polos)}")

## 6. Cruzar por nome e medir a cobertura

O CSV não traz o código do IBGE, então a junção é feita pelo **nome normalizado**: sem acento, sem pontuação, em minúsculas. Isso resolve `Restinga Sêca` × `Restinga Seca` e `Xangri-lá` × `Xangri-La`, mas **não** resolve grafias historicamente diferentes — `Sant'Ana do Livramento` na malha contra `Santana do Livramento` no CSV. Esse caso vai numa tabela de equivalências explícita, para ficar auditável em vez de escondido numa regra genérica.

A junção é `m:1` validada: cada município da malha casa com no máximo uma linha de região. Municípios sem correspondência recebem `NaN` — **não** são convertidos em uma categoria "outros", porque não sabemos a que região pertencem.

In [ ]:
def normalizar(texto):
    """Sem acento, sem pontuação, minúsculo — para casar nomes entre fontes."""
    texto = unicodedata.normalize("NFKD", str(texto).replace("'", " ").replace("’", " "))
    texto = texto.encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()


# Grafias que a normalização não reconcilia sozinha: chave = nome no CSV.
EQUIVALENCIAS = {"santana do livramento": "sant ana do livramento"}

municipios["chave"] = municipios["NM_MUN"].map(normalizar)
polos["chave"] = polos["nome_csv"].map(normalizar).replace(EQUIVALENCIAS)

sem_par = sorted(set(polos["chave"]) - set(municipios["chave"]))
if sem_par:
    raise ValueError(f"Nomes do CSV sem correspondência na malha: {sem_par}. "
                     "Acrescente-os a EQUIVALENCIAS em vez de ignorá-los.")

classificados = municipios.merge(
    polos[["chave", "rf", "corede"]], on="chave", how="left", validate="m:1"
)
if len(classificados) != len(municipios):
    raise ValueError("A junção alterou a quantidade de municípios.")

NOMES_RF = {
    1: "RF1 · Metropolitana", 2: "RF2 · Vales", 3: "RF3 · Sul e Litoral",
    4: "RF4 · Fronteira Oeste", 5: "RF5 · Central", 6: "RF6 · Noroeste e Missões",
    7: "RF7 · Produção", 8: "RF8 · Serra", 9: "RF9 · Norte",
}
classificados["regiao_funcional"] = classificados["rf"].map(NOMES_RF)
classificados["tem_regiao"] = classificados["rf"].notna()

com = int(classificados["tem_regiao"].sum())
print(f"Municípios com Região Funcional: {com} de {len(classificados)} "
      f"({100 * com / len(classificados):.1f}%)")
print(f"Sem correspondência no arquivo:  {len(classificados) - com}")

### Cobertura por região

A tabela abaixo é o diagnóstico que sustenta a ressalva da seção 1: mostra quantos municípios cada Região Funcional tem nomeados no arquivo e quanto do território estadual isso representa.

A área é calculada em **UTM estimada para o estado**, e não no CRS geográfico original. `estimate_utm_crs` escolhe a zona a partir da extensão dos dados; para um território do tamanho do RS isso é uma aproximação razoável, mas não substitui uma projeção equivalente em área para medições oficiais.

In [ ]:
CRS_AREA = classificados.estimate_utm_crs()
if CRS_AREA is None:
    raise ValueError("Não foi possível estimar a projeção UTM.")
classificados["area_km2"] = classificados.to_crs(CRS_AREA).area / 1_000_000
print("Projeção usada para área:", CRS_AREA.name)

cobertura = (
    classificados[classificados["tem_regiao"]]
    .groupby("regiao_funcional")
    .agg(municipios=("CD_MUN", "size"),
         coredes=("corede", "nunique"),
         area_km2=("area_km2", "sum"))
    .sort_index()
)
cobertura["% da área do RS"] = 100 * cobertura["area_km2"] / classificados["area_km2"].sum()

print(f"\nCobertura por Região Funcional (de {len(classificados)} municípios do estado):")
display(cobertura.round(1))

area_coberta = 100 * classificados.loc[classificados["tem_regiao"], "area_km2"].sum() / classificados["area_km2"].sum()
print(f"Território do RS classificado: {area_coberta:.1f}% da área")

## 7. Dissolver: de municípios para regiões

`dissolve` funde os polígonos dos municípios de cada região num único registro. Como só os polos estão classificados, cada Região Funcional vira um **multipolígono descontínuo** — um conjunto de manchas separadas, e não um bloco contíguo.

Isso não é um defeito do código: é a representação correta do que o arquivo permite afirmar. Um bloco contíguo exigiria conhecer a região dos 413 municípios ausentes.

In [ ]:
regioes_geo = (
    classificados[classificados["tem_regiao"]]
    .dissolve(by="regiao_funcional", aggfunc={"CD_MUN": "count", "area_km2": "sum"})
    .rename(columns={"CD_MUN": "municipios"})
    .reset_index()
)
regioes_geo["partes"] = regioes_geo.geometry.apply(
    lambda g: len(g.geoms) if g.geom_type == "MultiPolygon" else 1
)

print(f"Regiões Funcionais representadas: {len(regioes_geo)} de 9")
display(regioes_geo.drop(columns="geometry").round(1))
print("\n'partes' = manchas desconexas de cada região; reflete a cobertura parcial, não erro de geometria.")

## 8. Salvar os dados preparados

O GeoPackage guarda as duas camadas — municípios classificados e regiões dissolvidas — no mesmo arquivo, sem simplificar a geometria. O CSV traz só os atributos. O GeoJSON vai em `EPSG:4326` (longitude/latitude) e com simplificação de 200 m, adequada à escala estadual e suficiente para reduzir o peso do mapa web.

`to_crs` **transforma** coordenadas; `set_crs` apenas declara o sistema de referência. Trocar o rótulo não reprojeta nada.

In [ ]:
SAIDA_GPKG = PASTA_PREPARADOS / "regioes_funcionais_rs.gpkg"
SAIDA_CSV = PASTA_PREPARADOS / "municipios_classificados.csv"
SAIDA_GEOJSON = PASTA_PREPARADOS / "municipios_web.geojson"

colunas_saida = ["CD_MUN", "NM_MUN", "regiao_funcional", "corede", "tem_regiao", "area_km2", "geometry"]
municipios_saida = classificados[colunas_saida]

# mode="a" ACRESCENTA linhas a uma camada existente. Sem apagar o arquivo antes,
# rodar o notebook duas vezes duplicaria a camada de regiões. Apagar e reescrever
# mantém a célula idempotente.
SAIDA_GPKG.unlink(missing_ok=True)
municipios_saida.to_file(SAIDA_GPKG, layer="municipios", driver="GPKG", mode="w", index=False)
regioes_geo.to_file(SAIDA_GPKG, layer="regioes_funcionais", driver="GPKG", mode="a", index=False)
municipios_saida.drop(columns="geometry").to_csv(SAIDA_CSV, index=False, encoding="utf-8")

web = municipios_saida.to_crs(CRS_AREA).copy()
web["geometry"] = web.geometry.simplify(200, preserve_topology=True)
web = web.to_crs(4326)
web.to_file(SAIDA_GEOJSON, driver="GeoJSON", index=False)

for arquivo in (SAIDA_GPKG, SAIDA_CSV, SAIDA_GEOJSON):
    print(f"{arquivo.name:34s} {arquivo.stat().st_size / 1_048_576:6.2f} MB")

### Registrar fontes e decisões

Um nome de arquivo não diz qual edição foi usada nem que escolhas foram feitas no caminho. O registro abaixo acompanha os dados e guarda o que um leitor precisaria para reproduzir ou contestar o resultado — inclusive a cobertura parcial e os municípios excluídos por ambiguidade.

In [ ]:
def descrever_fonte(arquivo, url):
    registro = arquivo.with_suffix(arquivo.suffix + ".fonte.json")
    meta = json.loads(registro.read_text(encoding="utf-8")) if registro.exists() else {}
    return {"arquivo": arquivo.name, "url": url, "sha256": sha256(arquivo),
            "bytes": arquivo.stat().st_size, "baixado_em_utc": meta.get("baixado_em_utc")}


metadados = {
    "objetivo": "Separar os municípios do RS pelas Regiões Funcionais de Planejamento",
    "preparado_em_utc": datetime.now(timezone.utc).isoformat(),
    "malha": {"ano": 2022, "crs_original": str(malha.crs), "crs_area": str(CRS_AREA)},
    "cobertura": {
        "municipios_no_estado": int(len(classificados)),
        "com_regiao_funcional": int(classificados["tem_regiao"].sum()),
        "sem_correspondencia": int((~classificados["tem_regiao"]).sum()),
        "percentual_da_area_classificada": round(float(area_coberta), 1),
    },
    "decisoes": [
        "Removidas as feições 4300001 e 4300002 (Lagoa Mirim e Lagoa dos Patos): não são municípios.",
        "Junção por nome normalizado (sem acento/pontuação), pois o CSV não traz código do IBGE.",
        f"Equivalências de grafia aplicadas: {EQUIVALENCIAS}.",
        "Linhas marcadas '(parcial)' não valem como atribuição primária de COREDE.",
        f"Excluídos por Região Funcional ambígua: {ambiguos}.",
        "Municípios sem correspondência permanecem sem região; não foram atribuídos por proximidade.",
    ],
    "fontes": [
        descrever_fonte(ARQUIVO_MALHA, URL_MALHA),
        descrever_fonte(ARQUIVO_REGIOES, URL_REGIOES),
    ],
}

SAIDA_META = PASTA_PREPARADOS / "fontes_e_preparo.json"
SAIDA_META.write_text(json.dumps(metadados, ensure_ascii=False, indent=2), encoding="utf-8")
print(SAIDA_META.read_text(encoding="utf-8")[:900], "...")

## 9. Reabrir e conferir

Salvar sem testar a leitura esconde problemas de coluna, codificação e formato. Reabrimos os três arquivos e comparamos contagem, códigos, área e CRS com o que está em memória.

O CSV não carrega esquema de tipos: ao reabrir, o código do município precisa ser declarado como texto, ou `4300034` volta como número.

In [ ]:
mun_reaberto = gpd.read_file(SAIDA_GPKG, layer="municipios")
rf_reaberto = gpd.read_file(SAIDA_GPKG, layer="regioes_funcionais")
csv_reaberto = pd.read_csv(SAIDA_CSV, dtype={"CD_MUN": str})
web_reaberto = gpd.read_file(SAIDA_GEOJSON)

assert len(mun_reaberto) == len(csv_reaberto) == len(web_reaberto) == len(classificados)
assert set(mun_reaberto["CD_MUN"]) == set(classificados["CD_MUN"])
assert len(rf_reaberto) == len(regioes_geo)
assert mun_reaberto.crs == classificados.crs
assert web_reaberto.crs.to_epsg() == 4326
assert abs(mun_reaberto["area_km2"].sum() - classificados["area_km2"].sum()) < 1

print("Leitura validada: GPKG (2 camadas), CSV e GeoJSON.")
print(f"  municípios: {len(mun_reaberto)} | regiões: {len(rf_reaberto)}")
print(f"  área total: {mun_reaberto['area_km2'].sum():,.0f} km²".replace(",", "."))

## 10. Conferência visual

O mapa é montado a partir do **GeoJSON reaberto**, o que demonstra que outro notebook pode consumir a saída sem refazer download nem junção.

Leitura do mapa: cada cor é uma Região Funcional; o **cinza** são os 413 municípios sem correspondência no arquivo. O cinza dominante é o resultado honesto da cobertura de 17% dos municípios, e não uma falha de renderização.

In [ ]:
CORES_RF = {
    "RF1 · Metropolitana": "#e6550d", "RF2 · Vales": "#31a354",
    "RF3 · Sul e Litoral": "#3182bd", "RF4 · Fronteira Oeste": "#756bb1",
    "RF5 · Central": "#e7ba52", "RF6 · Noroeste e Missões": "#17becf",
    "RF7 · Produção": "#d6616b", "RF8 · Serra": "#8c6d31",
    "RF9 · Norte": "#c994c7",
}
CINZA = "#d9d9d9"


def estilo(feicao):
    regiao = feicao["properties"].get("regiao_funcional")
    return {"fillColor": CORES_RF.get(regiao, CINZA), "color": "white",
            "weight": 0.4, "fillOpacity": 0.85 if regiao else 0.45}


mapa = folium.Map(tiles="CartoDB positron", control_scale=True, height=620)
folium.GeoJson(
    web_reaberto.__geo_interface__, name="Municípios", style_function=estilo,
    tooltip=folium.GeoJsonTooltip(
        fields=["NM_MUN", "regiao_funcional", "corede", "area_km2"],
        aliases=["Município:", "Região Funcional:", "COREDE:", "Área (km²):"],
        localize=True),
).add_to(mapa)

oeste, sul, leste, norte = web_reaberto.total_bounds
mapa.fit_bounds([[sul, oeste], [norte, leste]])

legenda = "".join(
    f'<div><span style="background:{cor};width:12px;height:12px;'
    f'display:inline-block;margin-right:6px"></span>{nome}</div>'
    for nome, cor in CORES_RF.items()
)
legenda += (f'<div style="margin-top:4px"><span style="background:{CINZA};width:12px;height:12px;'
            'display:inline-block;margin-right:6px"></span>Sem correspondência no arquivo</div>')
mapa.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
    'padding:10px 12px;border:1px solid #ccc;border-radius:4px;font:12px sans-serif">'
    '<b>Regiões Funcionais de Planejamento — RS</b>' + legenda + '</div>'
))

# Salva o mapa como HTML autocontido, para abrir no navegador sem rodar o
# notebook de novo. O arquivo precisa de internet para o mapa-base e para as
# bibliotecas JavaScript do Leaflet.
SAIDA_MAPA = PASTA_PREPARADOS / "mapa_regioes_funcionais.html"
mapa.save(SAIDA_MAPA)
print("Mapa salvo em:", SAIDA_MAPA.resolve())
print(f"  {SAIDA_MAPA.stat().st_size / 1_048_576:.1f} MB — abra com duplo clique ou arraste para o navegador")

mapa

## 11. O que fazer com isto

### Como visualizar

| Quero... | Como |
|---|---|
| Ver o mapa sem rodar nada | Abrir `dados/regioes_funcionais/preparados/mapa_regioes_funcionais.html` no navegador |
| Ver o mapa aqui no notebook | Executar a seção 10 — o mapa aparece embaixo da célula |
| Inspecionar as geometrias | Arrastar `municipios_web.geojson` para [geojson.io](https://geojson.io) |
| Abrir num SIG | `regioes_funcionais_rs.gpkg` no QGIS; as duas camadas aparecem na lista |
| Só os atributos | `municipios_classificados.csv` em qualquer planilha |

O HTML é autocontido quanto aos dados, mas busca o mapa-base e as bibliotecas do Leaflet na internet — sem conexão, os polígonos aparecem sobre fundo branco. No Jupyter, se o mapa não renderizar, marque o notebook como confiável (*Trust Notebook*).

### Continuar a análise

Os produtos estão em `dados/regioes_funcionais/preparados/`. Para continuar em outro notebook:

```python
import geopandas as gpd
municipios = gpd.read_file(
    "dados/regioes_funcionais/preparados/regioes_funcionais_rs.gpkg",
    layer="municipios"
)
regioes = gpd.read_file(
    "dados/regioes_funcionais/preparados/regioes_funcionais_rs.gpkg",
    layer="regioes_funcionais"
)
```

Para juntar indicadores socioeconômicos, use `CD_MUN` como chave sempre que a outra fonte tiver código do IBGE — é mais seguro que nome. O `dataset_principal.csv` e o `pib_municipios_rs_2010.csv` deste repositório se ligam pela coluna `Territorialidades`, que é o nome do município seguido de `(RS)`.

### Limitações

- **Cobertura de 84 dos 497 municípios (17%, 39% da área).** É a restrição dominante, e vem do arquivo de origem, não do processamento. Qualquer estatística por região calculada a partir daqui descreve os **polos**, não as regiões.
- **Junção por nome.** Sem código do IBGE, a junção depende de grafia. O código falha explicitamente se um nome do CSV não encontrar par, em vez de descartar em silêncio — mas nomes iguais para municípios diferentes passariam despercebidos.
- **Frederico Westphalen excluído** por aparecer em Regiões Funcionais distintas no arquivo.
- **Malha de 2022, arquivo de regiões sem ano declarado.** Se a composição dos COREDEs mudou entre as duas datas, a diferença não é detectável aqui.
- **Área em UTM estimada.** Adequada para comparação relativa; para medida oficial, use a `AREA_KM2` publicada pelo IBGE na própria malha.

### Exercícios

1. Compare a `area_km2` calculada com a coluna `AREA_KM2` da malha original. As diferenças vêm da projeção escolhida — quantifique-as.
2. Junte o `pib_municipios_rs_2010.csv` e calcule o PIB per capita mediano dos polos de cada Região Funcional. Explique por que o resultado não pode ser lido como "PIB da região".
3. Obtenha o mapeamento completo município → COREDE na SEPLAG-RS, refaça a junção e compare a área classificada com os 39% atuais.